# Object mention priority diagnostic

CPU only. Attach the same three datasets as the spatial prompt audit. Send object_priority_train2000.json.

In [ ]:
from pathlib import Path
import numpy as np
def detection_index(source):
    if isinstance(source, dict) and 'data' in source:
        source = source['data']
    if not isinstance(source, dict):
        raise ValueError('Detector source must map filenames to records')
    result = {}
    for name, record in source.items():
        filename = Path(name).name
        if filename in result:
            raise ValueError('Duplicate detector basename: ' + filename)
        objects = record if isinstance(record, list) else record['objects']
        if not isinstance(objects, list):
            raise ValueError('Missing/malformed objects: ' + filename)
        for obj in objects:
            box = np.asarray(obj['bbox'], dtype=float)
            score = float(obj['conf'])
            if box.shape != (4,) or not np.isfinite(box).all() or (not np.isfinite(score)) or (not 0 <= score <= 1):
                raise ValueError('Nonfinite/invalid detector value: ' + filename)
            if box[2] <= box[0] or box[3] <= box[1]:
                raise ValueError('Nonpositive xyxy area: ' + filename)
        result[filename] = objects
    return result
"""Rule-based COCO object consistency scoring for caption candidates."""
import re
from pathlib import PurePosixPath


COCO_LABELS = (
    'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck',
    'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench',
    'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra',
    'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee',
    'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove',
    'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup',
    'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange',
    'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch',
    'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse',
    'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink',
    'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear',
    'hair drier', 'toothbrush',
)

SPECIAL_ALIASES = {
    'person': ('people', 'man', 'men', 'woman', 'women', 'boy', 'boys', 'girl',
               'girls', 'child', 'children', 'kid', 'kids', 'rider', 'riders'),
    'bicycle': ('bike', 'bikes'),
    'car': ('cars', 'vehicle', 'vehicles', 'automobile', 'automobiles'),
    'motorcycle': ('motorbike', 'motorbikes'),
    'airplane': ('plane', 'planes', 'aircraft', 'jet', 'jets'),
    'bus': ('buses',),
    'boat': ('ship', 'ships'),
    'traffic light': ('traffic signal', 'traffic signals'),
    'handbag': ('purse', 'purses'),
    'suitcase': ('luggage',),
    'sports ball': ('ball', 'balls'),
    'baseball glove': ('glove', 'gloves'),
    'tennis racket': ('racket', 'rackets', 'racquet', 'racquets'),
    'couch': ('sofa', 'sofas'),
    'potted plant': ('plant', 'plants'),
    'dining table': ('table', 'tables'),
    'tv': ('television', 'televisions', 'monitor', 'monitors'),
    'cell phone': ('phone', 'phones', 'mobile phone', 'mobile phones'),
    'refrigerator': ('fridge', 'fridges'),
    'donut': ('doughnut', 'doughnuts'),
    'teddy bear': ('stuffed bear', 'stuffed bears'),
    'hair drier': ('hair dryer', 'hair dryers'),
}


def _pluralize(phrase):
    words = phrase.split()
    last = words[-1]
    if last.endswith('y') and len(last) > 1 and last[-2] not in 'aeiou':
        words[-1] = last[:-1] + 'ies'
    elif last.endswith(('s', 'x', 'ch', 'sh')):
        words[-1] = last + 'es'
    else:
        words[-1] = last + 's'
    return ' '.join(words)


def _alias_table():
    aliases = []
    for label in COCO_LABELS:
        values = {label, _pluralize(label), *SPECIAL_ALIASES.get(label, ())}
        for value in values:
            aliases.append((tuple(value.split()), label))
    return sorted(aliases, key=lambda item: (-len(item[0]), -len(' '.join(item[0]))))


ALIASES = _alias_table()


def extract_caption_objects(caption):
    """Return non-overlapping COCO classes explicitly mentioned in a caption."""
    tokens = re.findall(r"[a-z0-9]+", caption.lower())
    occupied, labels = set(), []
    for alias, label in ALIASES:
        width = len(alias)
        for start in range(len(tokens) - width + 1):
            positions = set(range(start, start + width))
            if positions & occupied or tuple(tokens[start:start + width]) != alias:
                continue
            occupied.update(positions)
            if label not in labels:
                labels.append(label)
    return labels



import hashlib, json, math
from collections import Counter
ROOT=Path('/kaggle/input/datasets')
COCO=ROOT/'vuthetam/mscoco-2014'
OUT=Path('/kaggle/working/object_priority_train2000')
def checked(root,name,digest):
    paths=list(root.rglob(name))
    if len(paths)!=1:
        raise ValueError(f'Expected one {name} under {root}, found {len(paths)}')
    with paths[0].open('rb') as f:
        if hashlib.file_digest(f,'sha256').hexdigest()!=digest:
            raise ValueError(f'SHA256 mismatch: {paths[0]}')
    return paths[0]


OUT=OUT.with_suffix('.json')
COUNT=2000
COCO_SHA='2fd999220673258012acfb411a4e7e66af7d488050b2519b0badcc49b7600b8d'
YOLO_SHA='498825daf8f8773a1c92f4111dc4ccf1f4296343f952b486e0aa56caf82ec04c'
PROMPT_SHA='e90c79b3b04540630b4bd2d279a4070ef05da7f4b65970c952851c37e2423bf0'
def priority(objects, slots, mentions):
    ordered=sorted(enumerate(objects),key=lambda pair:(-float(pair[1]['conf']),pair[0]))[:150]
    accepted={}
    for roi,(_,obj) in enumerate(ordered):
        label=' '.join(obj['label'].lower().replace('_',' ').split())
        if label in COCO_LABELS and float(obj['conf'])>=.5:
            accepted[roi]=(label,float(obj['conf']))
    multiplicity=Counter(label for label,_ in accepted.values())
    result=[]
    for slot in slots:
        roi,label=slot['roi'],slot['label']
        value=accepted.get(roi)
        if value is None:
            continue
        if value[0]!=label:
            raise ValueError(f'Detector/slot label mismatch at ROI {roi}')
        if multiplicity[label]!=1:
            continue
        x0,y0,x1,y1=slot['roi_bbox']
        if not (0<=x0<x1<=1 and 0<=y0<y1<=1):
            raise ValueError(f'Invalid normalized ROI {roi}')
        result.append(dict(roi=roi,label=label,confidence=value[1],area=(x1-x0)*(y1-y0),mentions=mentions[label]))
    return result

def dcg(rows,key):
    ordered=sorted(rows,key=lambda r:(-r[key],r['roi']))[:3]
    return sum(r['mentions']/math.log2(i+2) for i,r in enumerate(ordered))

assert set(extract_caption_objects('Two men ride bicycles next to a hot dog stand.'))=={'person','bicycle','hot dog'}
assert dcg([dict(roi=0,mentions=0,confidence=.9),dict(roi=1,mentions=4,confidence=.8)],'mentions') > dcg([dict(roi=0,mentions=0,confidence=.9),dict(roi=1,mentions=4,confidence=.8)],'confidence')

data=json.loads(checked(COCO,'dataset_coco.json',COCO_SHA).read_bytes())
rows=[r for r in data['images'] if r['split'] in ('train','restval')]
if len(rows)!=113287 or len({int(r['cocoid']) for r in rows})!=113287:
    raise ValueError('Expected exact Karpathy TRAIN pool')
rows=sorted(rows,key=lambda r:hashlib.sha256(f"object-priority/1411/{r['cocoid']}".encode()).digest())[:COUNT]
wanted={int(r['cocoid']):r for r in rows}
del data, rows
det=detection_index(json.loads(checked(ROOT/'ducanh2403/objectdetectionecache','objectdetectioncache.json',YOLO_SHA).read_bytes()))
links=checked(ROOT/'ducanh2403/csa-prompt-full-karpathy-v175','prompt_links.jsonl',PROMPT_SHA)
selected={}
with links.open(encoding='utf8') as f:
    for line in f:
        r=json.loads(line)
        cid=r['coco_id']
        if cid in wanted:
            if r['group']!='train' or r['filename']!=wanted[cid]['filename'] or cid in selected:
                raise ValueError(f'Prompt cache row mismatch at {cid}')
            selected[cid]=r['objects']
if selected.keys()!=wanted.keys():
    raise ValueError('Missing TRAIN prompt rows')
c=Counter()
records=[]
ndcg=Counter()
for cid,row in wanted.items():
    filename=row['filename']
    if filename not in det:
        c['missing_detector']+=1
        continue
    refs=row['sentences']
    if len(refs)!=5:
        c['non_five_references']+=1
        continue
    mentions=Counter(label for ref in refs for label in extract_caption_objects(ref['raw']))
    ranked=priority(det[filename],selected[cid],mentions)
    c['with_prompt']+=bool(selected[cid])
    c['with_eligible_slot']+=bool(ranked)
    c['slots']+=len(ranked)
    c['mentioned_slots']+=sum(r['mentions']>0 for r in ranked)
    if len({r['mentions'] for r in ranked})<2:
        continue
    c['informative_images']+=1
    target=dcg(ranked,'mentions')
    for key in ('confidence','area'):
        ndcg[key]+=dcg(ranked,key)/target
    strong=[r for r in ranked if r['mentions']>=3]
    omitted=[r for r in ranked if not r['mentions']]
    if strong and omitted:
        c['strong_vs_unmentioned_images']+=1
        c['confidence_inversion_images']+=any(a['confidence']<b['confidence'] for a in strong for b in omitted)
        c['area_inversion_images']+=any(a['area']<b['area'] for a in strong for b in omitted)
    records.append(dict(coco_id=cid,slots=ranked))
summary=dict(c)
summary['sampled_images']=COUNT
summary['mean_ndcg_at_3']={k:ndcg[k]/c['informative_images'] if c['informative_images'] else None for k in ('confidence','area')}
payload=dict(split='train_restval',input_sha256=dict(coco=COCO_SHA,yolo=YOLO_SHA,prompt=PROMPT_SHA),summary=summary,records=records)
OUT.write_text(json.dumps(payload,indent=2,allow_nan=False),encoding='utf8')
print(json.dumps(summary,indent=2))
print('SEND:',OUT)
